# 02 — Friends Biomechanics

Parses raw `.FIT` directories exported by friends, builds per-person parquet files, and generates biomechanics scatter plots — same format as `01_eda_training_signals.ipynb`.

**Outputs (per person):**
- `data/<name>/sessions.parquet` — one row per activity
- `data/<name>/records.parquet` — per-second time series
- `data/biomechanics_plots_<name>/` — one PNG per running session

In [1]:
import io, os, warnings
from pathlib import Path

import fitparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────────────────
FRIENDS = {
    'Ciulavu': r'C:\Users\vasiu\UBB\Garmin Export Data\Ciulavu',
    'Mircea':  r'C:\Users\vasiu\UBB\Garmin Export Data\Mircea',
}

DATA_ROOT = r'C:\Users\vasiu\UBB\Quantified-Strides\data'

# Update these per person if you know their actual max HR.
# Used only for TRIMP zone weighting — rough values are fine.
MAX_HR = {
    'Ciulavu': 195,
    'Mircea':  190,
}

ATL_TAU = 7
CTL_TAU = 42
SEMI_TO_DEG = 180.0 / (2 ** 31)

RUNNING_SPORTS = {'running', 'trail_running'}

print('Config ready.')

Config ready.


## 1. FIT Parsers

In [2]:
def strip_tz(dt):
    return dt.replace(tzinfo=None) if dt and hasattr(dt, 'tzinfo') and dt.tzinfo else dt


def session_get(session, *names):
    for n in names:
        try:
            v = session.get_value(n)
            if v is not None:
                return v
        except Exception:
            pass
    return None


def parse_session(fit) -> dict | None:
    sessions = list(fit.get_messages('session'))
    if not sessions:
        return None
    s = sessions[0]
    sg = lambda *n: session_get(s, *n)

    start = strip_tz(sg('start_time'))
    if not start:
        return None
    elapsed = sg('total_elapsed_time') or 0

    sport_raw = str(sg('sport') or 'unknown').lower().replace(' ', '_')
    sport_map = {
        'running': 'running', 'trail_running': 'trail_running',
        'cycling': 'cycling', 'mountain_biking': 'mountain_biking',
        'e_biking': 'cycling', 'fitness_equipment': 'training',
        'strength_training': 'strength_training', 'generic': 'training',
        'training': 'training', 'yoga': 'training',
        'alpine_skiing': 'alpine_skiing', 'snowboarding': 'snowboarding',
        'rock_climbing': 'rock_climbing', 'hiking': 'hiking',
        'swimming': 'swimming', 'soccer': 'soccer',
    }
    sport = sport_map.get(sport_raw, sport_raw)

    avg_cad = sg('avg_running_cadence', 'avg_cadence')

    return {
        'sport':             sport,
        'start_time':        start,
        'date':              pd.Timestamp(start.date()),
        'duration_s':        float(elapsed),
        'distance_m':        float(sg('total_distance') or 0),
        'total_ascent':      float(sg('total_ascent') or 0),
        'total_descent':     float(sg('total_descent') or 0),
        'avg_hr':            sg('avg_heart_rate'),
        'max_hr':            sg('max_heart_rate'),
        'calories':          sg('total_calories'),
        'avg_cadence':       float(avg_cad) if avg_cad is not None else None,
        'avg_power':         sg('avg_power'),
        'norm_power':        sg('normalized_power'),
        'avg_vo':            sg('avg_vertical_oscillation'),
        'avg_stance':        sg('avg_stance_time'),
        'avg_step_len':      sg('avg_step_length'),
        'avg_vr':            sg('avg_vertical_ratio'),
        'training_effect':   sg('total_training_effect'),
        'anaerobic_effect':  sg('total_anaerobic_training_effect'),
    }


def parse_records(fit, session_id: int) -> list[dict]:
    rows = []
    prev_alt = prev_dist = None
    for rec in fit.get_messages('record'):
        def rv(f):
            try:
                return rec.get_value(f)
            except Exception:
                return None

        ts = strip_tz(rv('timestamp'))
        if not ts:
            continue

        lat  = rv('position_lat')
        lon  = rv('position_long')
        alt  = rv('enhanced_altitude') or rv('altitude')
        dist = rv('distance')
        spd  = rv('enhanced_speed') or rv('speed')
        pace = (1000 / float(spd) / 60) if spd else None

        raw_cad = rv('cadence')
        frac    = rv('fractional_cadence')
        cadence = ((float(raw_cad) + float(frac or 0)) * 2) if raw_cad is not None else None

        vo = rv('vertical_oscillation')
        if vo is not None:
            vo = vo / 10.0  # mm → cm

        grad = None
        if alt is not None and prev_alt is not None:
            d_alt  = float(alt) - prev_alt
            d_dist = (float(dist) - prev_dist) if (dist and prev_dist) else (1000 / (pace * 60) if pace else None)
            if d_dist and d_dist > 0.5:
                grad = round(d_alt / d_dist * 100, 2)
        if alt  is not None: prev_alt  = float(alt)
        if dist is not None: prev_dist = float(dist)

        rows.append({
            'session_id':           session_id,
            'timestamp':            ts,
            'heart_rate':           rv('heart_rate'),
            'speed_ms':             float(spd) if spd else None,
            'pace_min_km':          pace,
            'cadence':              cadence,
            'altitude':             float(alt) if alt else None,
            'distance':             float(dist) if dist else None,
            'power':                rv('power'),
            'lat':                  float(lat) * SEMI_TO_DEG if lat else None,
            'lon':                  float(lon) * SEMI_TO_DEG if lon else None,
            'vertical_oscillation': vo,
            'vertical_ratio':       rv('vertical_ratio'),
            'stance_time':          rv('stance_time'),
            'step_length':          rv('step_length'),
            'gradient_pct':         grad,
        })
    return rows


print('Parsers defined.')

Parsers defined.


## 2. Parse FIT files for each friend

In [3]:
def build_dataset(name: str, fit_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Parse all .FIT files in fit_dir → (df_sessions, df_records)."""
    paths = list(Path(fit_dir).rglob('*.fit')) + list(Path(fit_dir).rglob('*.FIT'))
    total = len(paths)
    print(f'[{name}] Found {total} .FIT files — parsing...')

    all_sessions, all_records = [], []
    session_id = 0
    skipped = errors = 0

    for i, p in enumerate(paths):
        if i % 500 == 0 and i > 0:
            print(f'  {i}/{total}  ({session_id} activities so far)')
        try:
            data = p.read_bytes()

            fit1 = fitparse.FitFile(io.BytesIO(data))
            sess = parse_session(fit1)
            if not sess:
                skipped += 1
                continue

            fit2 = fitparse.FitFile(io.BytesIO(data))
            recs = parse_records(fit2, session_id)
            if not recs:
                skipped += 1
                continue

            sess['session_id'] = session_id
            sess['n_records']  = len(recs)
            all_sessions.append(sess)
            all_records.extend(recs)
            session_id += 1

        except Exception as e:
            errors += 1

    df_sessions = pd.DataFrame(all_sessions).sort_values('start_time').reset_index(drop=True)
    df_records  = pd.DataFrame(all_records)

    print(f'[{name}] Done — {len(df_sessions)} activities, {len(df_records):,} record rows  '
          f'(skipped {skipped} non-activity, {errors} errors)')
    if len(df_sessions) > 0:
        print(f'  Date range : {df_sessions["date"].min().date()} → {df_sessions["date"].max().date()}')
        print(f'  Sports     : {df_sessions["sport"].value_counts().to_dict()}')
    return df_sessions, df_records


datasets = {}
for name, fit_dir in FRIENDS.items():
    df_s, df_r = build_dataset(name, fit_dir)
    datasets[name] = {'sessions': df_s, 'records': df_r}
    print()

[Ciulavu] Found 3890 .FIT files — parsing...
  500/3890  (39 activities so far)
  1000/3890  (72 activities so far)
  1500/3890  (101 activities so far)
  2000/3890  (125 activities so far)
  2500/3890  (165 activities so far)
  3000/3890  (199 activities so far)
  3500/3890  (226 activities so far)
[Ciulavu] Done — 248 activities, 488,924 record rows  (skipped 3642 non-activity, 0 errors)
  Date range : 2025-07-31 → 2026-04-19
  Sports     : {'training': 168, 'running': 56, 'swimming': 18, 'walking': 2, 'soccer': 2, 'hiking': 2}

[Mircea] Found 9046 .FIT files — parsing...
  500/9046  (6 activities so far)
  1000/9046  (14 activities so far)
  1500/9046  (22 activities so far)
  2000/9046  (29 activities so far)
  2500/9046  (44 activities so far)
  3000/9046  (56 activities so far)
  3500/9046  (67 activities so far)
  4000/9046  (73 activities so far)
  4500/9046  (82 activities so far)
  5000/9046  (88 activities so far)
  5500/9046  (96 activities so far)
  6000/9046  (104 activit

## 3. Compute TRIMP + ATL/CTL/TSB

In [4]:
ZONE_BOUNDS  = [0.50, 0.60, 0.70, 0.80, 0.90, 1.01]
ZONE_WEIGHTS = [1.0, 1.5, 2.0, 3.0, 4.0]


def compute_trimp(session_id: int, df_records: pd.DataFrame, max_hr: int) -> float:
    hr = df_records.loc[df_records['session_id'] == session_id, 'heart_rate'].dropna()
    if hr.empty:
        return 0.0
    trimp = 0.0
    for i, w in enumerate(ZONE_WEIGHTS):
        lo = ZONE_BOUNDS[i]   * max_hr
        hi = ZONE_BOUNDS[i+1] * max_hr
        trimp += ((hr >= lo) & (hr < hi)).sum() / 60 * w
    return trimp


def attach_training_load(df_sessions: pd.DataFrame, df_records: pd.DataFrame,
                         max_hr: int) -> pd.DataFrame:
    df = df_sessions.copy()
    df['trimp'] = df['session_id'].apply(
        lambda sid: compute_trimp(sid, df_records, max_hr)
    )
    # Fallback for sessions with no HR data
    no_hr = df['trimp'] == 0
    df.loc[no_hr, 'trimp'] = df.loc[no_hr, 'duration_s'] / 60 * 2.5

    daily_trimp = df.groupby('date')['trimp'].sum().reset_index()
    full_range  = pd.DataFrame({'date': pd.date_range(df['date'].min(), pd.Timestamp.today())})
    daily       = full_range.merge(daily_trimp, on='date', how='left').fillna(0)

    a_atl = 1 - np.exp(-1 / ATL_TAU)
    a_ctl = 1 - np.exp(-1 / CTL_TAU)
    atl = ctl = 0.0
    atl_vals, ctl_vals = [], []
    for t in daily['trimp']:
        atl = a_atl * t + (1 - a_atl) * atl
        ctl = a_ctl * t + (1 - a_ctl) * ctl
        atl_vals.append(atl)
        ctl_vals.append(ctl)
    daily['atl'] = atl_vals
    daily['ctl'] = ctl_vals
    daily['tsb'] = daily['ctl'] - daily['atl']
    last = daily.iloc[-1]
    print(f'  ATL={last["atl"]:.1f}  CTL={last["ctl"]:.1f}  TSB={last["tsb"]:.1f}')
    return df


for name in FRIENDS:
    print(f'[{name}]')
    d = datasets[name]
    d['sessions'] = attach_training_load(d['sessions'], d['records'], MAX_HR[name])

print('\nTRIMP done.')

[Ciulavu]
  ATL=63.6  CTL=57.5  TSB=-6.2
[Mircea]
  ATL=37.3  CTL=66.2  TSB=28.8

TRIMP done.


## 4. Save Parquet files

In [5]:
for name in FRIENDS:
    out_dir = os.path.join(DATA_ROOT, name)
    os.makedirs(out_dir, exist_ok=True)

    s_path = os.path.join(out_dir, 'sessions.parquet')
    r_path = os.path.join(out_dir, 'records.parquet')

    datasets[name]['sessions'].to_parquet(s_path, index=False)
    datasets[name]['records'].to_parquet(r_path,  index=False)

    s_mb = os.path.getsize(s_path) / 1e6
    r_mb = os.path.getsize(r_path) / 1e6
    print(f'[{name}]  sessions: {len(datasets[name]["sessions"]):>5} rows  {s_mb:.1f} MB  '
          f'|  records: {len(datasets[name]["records"]):>9,} rows  {r_mb:.1f} MB')

print('\nParquet files saved.')

[Ciulavu]  sessions:   248 rows  0.0 MB  |  records:   488,924 rows  7.0 MB
[Mircea]  sessions:   164 rows  0.0 MB  |  records:   320,730 rows  7.4 MB

Parquet files saved.


## 5. Biomechanics plots — per running session

In [6]:
def plot_biomechanics(name: str, df_sessions: pd.DataFrame, df_records: pd.DataFrame):
    runs = df_sessions[df_sessions['sport'].isin(RUNNING_SPORTS)].copy()
    print(f'[{name}] {len(runs)} running sessions found')
    if runs.empty:
        print('  → no running data, skipping plots')
        return

    out_dir = os.path.join(DATA_ROOT, f'biomechanics_plots_{name}')
    os.makedirs(out_dir, exist_ok=True)

    run_records = df_records[df_records['session_id'].isin(runs['session_id'])]

    for _, row in runs.iterrows():
        sid  = row['session_id']
        date = row['date'].strftime('%Y-%m-%d')
        sr   = run_records[run_records['session_id'] == sid]

        # Skip sessions with no usable biomechanics data
        if sr[['speed_ms', 'cadence', 'vertical_oscillation', 'vertical_ratio']].dropna(how='all').empty:
            print(f'  Skipping {sid} {date} — no biomechanics data')
            continue

        dist_km = row['distance_m'] / 1000
        dur_min = row['duration_s'] / 60

        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        fig.suptitle(
            f'{name}  |  Run {sid} — {date}  '
            f'({dist_km:.1f} km, {dur_min:.0f} min)',
            fontsize=12
        )

        axes[0, 0].scatter(sr['speed_ms'], sr['cadence'], alpha=0.3, s=8)
        axes[0, 0].set_xlabel('Speed (m/s)')
        axes[0, 0].set_ylabel('Cadence (spm)')
        axes[0, 0].set_title('Speed vs Cadence')

        axes[0, 1].scatter(sr['pace_min_km'], sr['stance_time'], alpha=0.3, s=8)
        axes[0, 1].set_xlabel('Pace (min/km)')
        axes[0, 1].set_ylabel('Stance Time (ms)')
        axes[0, 1].set_title('Pace vs Ground Contact Time')

        axes[1, 0].scatter(sr['speed_ms'], sr['vertical_oscillation'], alpha=0.3, s=8)
        axes[1, 0].set_xlabel('Speed (m/s)')
        axes[1, 0].set_ylabel('Vertical Oscillation (cm)')
        axes[1, 0].set_title('Speed vs Vertical Oscillation')

        axes[1, 1].scatter(sr['speed_ms'], sr['vertical_ratio'], alpha=0.3, s=8)
        axes[1, 1].set_xlabel('Speed (m/s)')
        axes[1, 1].set_ylabel('Vertical Ratio (%)')
        axes[1, 1].set_title('Speed vs Vertical Ratio')

        plt.tight_layout()
        out_path = os.path.join(out_dir, f'run_{sid}_{date}.png')
        plt.savefig(out_path, dpi=120)
        plt.close()

    saved = len(list(Path(out_dir).glob('*.png')))
    print(f'  → {saved} plots saved to {out_dir}')


for name in FRIENDS:
    plot_biomechanics(name, datasets[name]['sessions'], datasets[name]['records'])
    print()

[Ciulavu] 56 running sessions found
  → 56 plots saved to C:\Users\vasiu\UBB\Quantified-Strides\data\biomechanics_plots_Ciulavu

[Mircea] 98 running sessions found
  → 98 plots saved to C:\Users\vasiu\UBB\Quantified-Strides\data\biomechanics_plots_Mircea



## 6. Quick summary comparison

In [7]:
print(f'{"Name":<12} {"Activities":>12} {"Running":>10} {"Total km":>10} '
      f'{"Avg run km":>12} {"Avg run pace":>14}')
print('-' * 72)

for name in FRIENDS:
    df_s = datasets[name]['sessions']
    df_r = datasets[name]['records']

    runs = df_s[df_s['sport'].isin(RUNNING_SPORTS)]
    total_km = df_s['distance_m'].sum() / 1000
    avg_run_km = runs['distance_m'].mean() / 1000 if len(runs) > 0 else 0

    # Avg pace from records (cleaner than session-level)
    run_recs = df_r[df_r['session_id'].isin(runs['session_id'])]
    avg_pace = run_recs['pace_min_km'].dropna()
    avg_pace = avg_pace[(avg_pace > 3) & (avg_pace < 20)]  # filter noise
    avg_pace_val = avg_pace.mean() if len(avg_pace) > 0 else float('nan')
    pace_str = f"{int(avg_pace_val)}:{int((avg_pace_val % 1) * 60):02d} /km" if not np.isnan(avg_pace_val) else 'N/A'

    print(f'{name:<12} {len(df_s):>12} {len(runs):>10} {total_km:>10.0f} '
          f'{avg_run_km:>12.1f} {pace_str:>14}')

Name           Activities    Running   Total km   Avg run km   Avg run pace
------------------------------------------------------------------------
Ciulavu               248         56        338          5.4       9:03 /km
Mircea                164         98        614          6.2       6:04 /km
